# Weight Capping and Distribution

This notebook demonstrates a weight capping algorithm that redistributes excess weight from capped items to remaining items proportionally.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

np.random.seed(42)  # For reproducibility

## Cap and Distribute Function

This function caps weights at a maximum value and redistributes the excess proportionally to uncapped items.

In [ ]:
def cap_and_distribute(weights, cap=0.09):
    weights = np.array(weights, dtype=float)
    total = weights.sum()
    if not np.isclose(total, 1.0):
        weights /= total

    indexed = list(enumerate(weights))
    indexed.sort(key=lambda x: x[1], reverse=True)

    capped = dict()

    i = 0
    while i < len(indexed):
        idx, w = indexed[i]
        if w <= cap:
            break

        capped[idx] = cap
        excess = w - cap
        remaining = indexed[i+1:]
        if not remaining:
            break
        total_remaining = sum(wt for _, wt in remaining)
        distributed = [(j, wt + excess * (wt / total_remaining)) for j, wt in remaining]
        indexed = indexed[:i+1] + distributed
        indexed[i+1:] = sorted(indexed[i+1:], key=lambda x: x[1], reverse=True)
        i += 1

    final = np.zeros(len(weights))
    for idx, w in indexed:
        if idx not in capped:
            capped[idx] = w
    for idx, w in capped.items():
        final[idx] = w
    final /= final.sum()
    return final.tolist()

## Generate Random Weights

Generate a random weight distribution with 75 items following a power-law-like distribution.

In [ ]:
# Generate random weights with exponential decay
n_items = 75
raw_weights = np.random.exponential(scale=2.0, size=n_items)
raw_weights = np.sort(raw_weights)[::-1]  # Sort descending

# Normalize to sum to 1
weights = raw_weights / raw_weights.sum()

print(f"Generated {len(weights)} random weights")
print(f"Sum of weights: {sum(weights):.6f}")
print(f"Max weight: {max(weights):.4f}")
print(f"Min weight: {min(weights):.4f}")

## Display Original Weights

In [ ]:
print("\nOriginal Weights (rounded to 4 decimals):")
pprint([round(w, 4) for w in weights])

## Apply Capping and Distribution

In [ ]:
cap_value = 0.09
capped_weights = cap_and_distribute(weights, cap=cap_value)

print(f"\nCapped Weights (cap={cap_value}, rounded to 4 decimals):")
pprint([round(w, 4) for w in capped_weights])

print(f"\nSum of capped weights: {sum(capped_weights):.6f}")
print(f"Max capped weight: {max(capped_weights):.4f}")
print(f"Number of weights at cap: {sum(1 for w in capped_weights if abs(w - cap_value) < 1e-6)}")

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Original weights bar chart
axes[0, 0].bar(range(len(weights)), weights, color='steelblue', alpha=0.7)
axes[0, 0].axhline(y=cap_value, color='r', linestyle='--', label=f'Cap = {cap_value}')
axes[0, 0].set_xlabel('Item Index')
axes[0, 0].set_ylabel('Weight')
axes[0, 0].set_title('Original Weights')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Capped weights bar chart
axes[0, 1].bar(range(len(capped_weights)), capped_weights, color='coral', alpha=0.7)
axes[0, 1].axhline(y=cap_value, color='r', linestyle='--', label=f'Cap = {cap_value}')
axes[0, 1].set_xlabel('Item Index')
axes[0, 1].set_ylabel('Weight')
axes[0, 1].set_title('Capped & Redistributed Weights')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Sorted comparison
sorted_orig = sorted(weights, reverse=True)
sorted_capped = sorted(capped_weights, reverse=True)
x = range(len(weights))
axes[1, 0].plot(x, sorted_orig, marker='o', label='Original', linewidth=2, markersize=4)
axes[1, 0].plot(x, sorted_capped, marker='s', label='Capped', linewidth=2, markersize=4)
axes[1, 0].axhline(y=cap_value, color='r', linestyle='--', label=f'Cap = {cap_value}')
axes[1, 0].set_xlabel('Rank')
axes[1, 0].set_ylabel('Weight')
axes[1, 0].set_title('Sorted Weights Comparison')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Plot 4: Difference (capped - original)
difference = np.array(capped_weights) - np.array(weights)
colors = ['green' if d > 0 else 'red' for d in difference]
axes[1, 1].bar(range(len(difference)), difference, color=colors, alpha=0.7)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 1].set_xlabel('Item Index')
axes[1, 1].set_ylabel('Weight Change')
axes[1, 1].set_title('Change in Weights (Capped - Original)')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(f"Items that gained weight: {sum(1 for d in difference if d > 0)}")
print(f"Items that lost weight: {sum(1 for d in difference if d < 0)}")
print(f"Total weight redistributed: {sum(abs(d) for d in difference if d < 0):.6f}")